# LASTRO — segmentação experimental de distúrbio florestal
1. Envie este notebook ao Google Colab e selecione uma GPU T4, L4 ou A100.
2. Opcionalmente edite `AOIS`; o exemplo incluído funciona sem edição.
3. Clique em **Executar tudo**; autentique sua conta quando `ee.Authenticate()` abrir o fluxo GEE.
4. Use uma conta com acesso ao projeto `project-f5ab0bff-aa9b-4cb2-97d`.
5. `MODO_RAPIDO=True` usa 400 chips/até 10 épocas; duração depende da GPU e da rede.
6. Ao final, baixe `lastro_dl_resultados.zip`; cache opcional no Drive pede autorização adicional.
Resultados são indícios para priorizar fiscalização, nunca prova.

In [ ]:
# Instalar sem alterar a instalação CUDA/PyTorch fornecida pelo Colab.
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'earthengine-api','rasterio','geopandas','shapely>=2','pyproj','requests',
    'scikit-image','scikit-learn','lightgbm','matplotlib','segmentation-models-pytorch==0.5.0'])
TERRATORCH_INSTALL_ERROR = None
try:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','terratorch==1.2.10'])
except Exception as exc:
    TERRATORCH_INSTALL_ERROR = str(exc)
    print('TerraTorch indisponível; fallback U-Net será usado:',exc)


## Configuração

In [ ]:
from pathlib import Path
import os, json, random, time, hashlib, math, re
import numpy as np

MODO_RAPIDO = True
SEED = 42
N_CHIPS = 400 if MODO_RAPIDO else 1000
EPOCAS = 10 if MODO_RAPIDO else 20
WORKERS = 8
SALVAR_DRIVE = False  # True solicita também a autorização do Google Drive.
PROJECT = 'project-f5ab0bff-aa9b-4cb2-97d'
BBOX = [-66, -10, -60, -6.5]
ANOS = [2022, 2023, 2024]
BANDAS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
SIZE, RES, BLOCK = 224, 10, 25000
ROOT = Path('/content')
CACHE = ROOT / 'chips'
OUT = ROOT / 'lastro_dl_resultados'
for p in (CACHE, OUT):
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
AOIS = [{"autex": "EXEMPLO-AMACRO", "bbox": [-63.12,-7.64,-63.05,-7.58],
         "pre": ["2023-06-01","2023-09-30"], "pos": ["2024-06-01","2024-09-30"]}]
CONFIG = dict(seed=SEED, modo_rapido=MODO_RAPIDO, n_chips_alvo=N_CHIPS,
              epocas_max=EPOCAS, workers=WORKERS, bbox=BBOX, anos=ANOS,
              bandas=BANDAS, chip_px=SIZE, resolucao_m=RES, bloco_m=BLOCK,
              projeto=PROJECT, aois=AOIS, cloud_probability_lt=40,
              split=[0.70,0.15,0.15], threshold=0.5)

## Funções de dados (também usadas no teste local)

In [ ]:
import requests
import ee
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from pyproj import Transformer
from shapely.geometry import shape, box, mapping
from shapely.ops import transform as geom_transform
from shapely import make_valid
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

WFS = 'https://terrabrasilis.dpi.inpe.br/geoserver/ows'
CLASSES = ['CS_DESORDENADO','CS_GEOMETRICO','DEGRADACAO','DESMATAMENTO_CR']

def retry(fn, attempts=4):
    for attempt in range(attempts):
        try:
            return fn()
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)

def wfs_features(layer, bbox, condition, cache_dir=CACHE):
    """Paginação estável; erro de rede nunca vira conjunto vazio de rótulos."""
    clause = f"BBOX(geom,{','.join(map(str,bbox))},'EPSG:4326') AND ({condition})"
    key = hashlib.sha256((layer + clause).encode()).hexdigest()[:20]
    path = cache_dir / f'wfs_{key}.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    features, seen, start = [], set(), 0
    while True:
        params = dict(service='WFS', version='2.0.0', request='GetFeature',
                      typeNames=layer, outputFormat='application/json',
                      srsName='EPSG:4326', CQL_FILTER=clause,
                      count=2000, startIndex=start, sortBy='fid' if 'prodes' in layer else 'gid')
        def fetch():
            r = requests.get(WFS, params=params, timeout=(20,180))
            r.raise_for_status()
            data = r.json()
            if 'features' not in data:
                raise RuntimeError(str(data)[:500])
            return data
        data = retry(fetch)
        page = data['features']
        if not page:
            break
        fresh = [f for f in page if f['id'] not in seen]
        if not fresh:
            raise RuntimeError('WFS repetiu a página: paginação não confiável.')
        features.extend(fresh)
        seen.update(f['id'] for f in fresh)
        start += len(page)
        total = data.get('numberMatched', data.get('totalFeatures'))
        if isinstance(total, int) and start >= total:
            break
        if len(page) < 2000:
            break
    path.write_text(json.dumps(features), encoding='utf-8')
    return features

def load_labels(bbox, years=ANOS):
    # Histórico também exclui negativos em áreas de distúrbio conhecido.
    prodes = wfs_features('prodes-legal-amz:yearly_deforestation', bbox,
                         f'year >= 2008 AND year <= {max(years)}')
    classes = ','.join("'" + c + "'" for c in CLASSES)
    deter = wfs_features('deter-amz:deter_amz', bbox,
                        f"view_date >= '2008-01-01' AND view_date < '{max(years)+1}-01-01'")
    labels = {}
    for year in years:
        labels[year] = [f for f in prodes if int(f['properties']['year']) == year]
        labels[year] += [f for f in deter
                        if f['properties'].get('classname') in CLASSES
                        and f'{year-1}-10-01' <= str(f['properties'].get('view_date',''))[:10] <= f'{year}-09-30']
    return labels, prodes + deter

def utm_epsg(lon):
    return 31980 if lon < -60 else 31981

def rasterize_labels(features, transform, crs, size=SIZE):
    bounds = rasterio.transform.array_bounds(size, size, transform)
    footprint = box(*bounds)
    project = Transformer.from_crs('EPSG:4326', crs, always_xy=True).transform
    geometries = []
    for feature in features:
        if not feature.get('geometry'):
            continue
        g = geom_transform(project, make_valid(shape(feature['geometry'])))
        if g.intersects(footprint):
            geometries.append((mapping(g), 1))
    return rasterize(geometries, out_shape=(size,size), transform=transform,
                     fill=0, dtype='uint8') if geometries else np.zeros((size,size), np.uint8)

def composite(start, end, region):
    # Datas finais inclusivas na interface, exclusivas no Earth Engine.
    end_exclusive = (date.fromisoformat(end) + timedelta(days=1)).isoformat()
    sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(region).filterDate(start,end_exclusive)
    clouds = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filterBounds(region).filterDate(start,end_exclusive)
    joined = ee.Join.saveFirst('clouds').apply(sr, clouds, ee.Filter.equals(leftField='system:index', rightField='system:index'))
    def clean(im):
        im = ee.Image(im)
        scl = im.select('SCL')
        mask = ee.Image(im.get('clouds')).select('probability').lt(40)
        mask = mask.And(scl.neq(0)).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        # Máscaras de borda nas bandas de 20/60 m.
        mask = mask.And(im.select('B8A').mask()).And(im.select('B9').mask())
        return im.select(BANDAS).updateMask(mask).multiply(0.0001).toFloat()
    return ee.ImageCollection(joined).map(clean).median()

def stack_image(pre, pos, region):
    a, b = composite(*pre,region), composite(*pos,region)
    names = ['pre_'+x for x in BANDAS] + ['pos_'+x for x in BANDAS]
    im = a.addBands(b).rename(names)
    valid = im.mask().reduce(ee.Reducer.min()).rename('valid')
    return im.unmask(-9999).addBands(valid.unmask(0)).toFloat()

def download_chip(image, x, y, epsg, size=SIZE):
    """x,y = canto superior esquerdo UTM; retorno CHW, máscara e affine."""
    result = retry(lambda: ee.data.computePixels(dict(
        expression=image, fileFormat='NUMPY_NDARRAY',
        grid=dict(dimensions=dict(width=size,height=size),
                  affineTransform=dict(scaleX=RES,shearX=0,translateX=float(x),
                                       shearY=0,scaleY=-RES,translateY=float(y)),
                  crsCode=f'EPSG:{epsg}'))))
    bands = ['pre_'+b for b in BANDAS] + ['pos_'+b for b in BANDAS]
    arr = np.stack([result[b] for b in bands]).astype('float32')
    valid = (result['valid'] > 0) & np.isfinite(arr).all(axis=0) & (arr != -9999).all(axis=0)
    arr[:,~valid] = 0
    return arr, valid, from_origin(x,y,RES,RES)

def ndvi(x, offset=0):
    return (x[offset+3]-x[offset+2]) / (x[offset+3]+x[offset+2]+1e-6)

def block_id(x,y,epsg=31980):
    return f'{epsg}_{math.floor(x/BLOCK)}_{math.floor(y/BLOCK)}'

def candidate(lon,lat,year):
    epsg = utm_epsg(lon)
    x,y = Transformer.from_crs(4326,epsg,always_xy=True).transform(lon,lat)
    # Mantém footprint inteiro dentro de um único bloco; sem sobreposição entre splits.
    bx,by = math.floor(x/BLOCK)*BLOCK, math.floor(y/BLOCK)*BLOCK
    left = max(bx, min(bx+BLOCK-SIZE*RES, math.floor((x-SIZE*RES/2)/RES)*RES))
    top = max(by+SIZE*RES, min(by+BLOCK, math.ceil((y+SIZE*RES/2)/RES)*RES))
    return dict(year=int(year), x=left,y=top,epsg=epsg,bloco=block_id(x,y,epsg))

def split_blocks(blocks):
    unique = sorted(set(blocks))
    if len(unique) < 7:
        raise RuntimeError('Menos de 7 blocos espaciais: ampliar a amostragem.')
    np.random.default_rng(SEED).shuffle(unique)
    ntest = max(1, round(len(unique)*.15))
    nval = max(1, round(len(unique)*.15))
    return {b: ('test' if i<ntest else 'val' if i<ntest+nval else 'train') for i,b in enumerate(unique)}

## Download e amostragem

In [ ]:
ee.Authenticate()
ee.Initialize(project='project-f5ab0bff-aa9b-4cb2-97d')
print('GEE autenticado.')

In [ ]:
labels, history = load_labels(BBOX)
print('Rótulos por ano:', {y:len(v) for y,v in labels.items()})
import geopandas as gpd
label_frames = {y:gpd.GeoDataFrame.from_features(fs,crs=4326) for y,fs in labels.items()}
history_frame = gpd.GeoDataFrame.from_features(history,crs=4326)
images = {y:stack_image([f'{y-1}-06-01',f'{y-1}-09-30'],
                        [f'{y}-06-01',f'{y}-09-30'], ee.Geometry.Rectangle(BBOX)) for y in ANOS}
rng = np.random.default_rng(SEED)
demo_boxes = [box(*a['bbox']) for a in AOIS]

def nearby(frame, bbox):
    indices = frame.sindex.query(box(*bbox), predicate='intersects')
    return json.loads(frame.iloc[indices].to_json())['features']

def materialize(c):
    key = hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()[:18]
    path = CACHE / f'chip_v3_{key}.npz'
    affine = from_origin(c['x'],c['y'],RES,RES)
    bounds = rasterio.transform.array_bounds(SIZE,SIZE,affine)
    geo = geom_transform(Transformer.from_crs(c['epsg'],4326,always_xy=True).transform,box(*bounds))
    if not box(*BBOX).covers(geo) or any(geo.intersects(d) for d in demo_boxes):
        return None
    if path.exists():
        with np.load(path) as z:
            positive, frac = bool(z['positive']), float(z['valid'].mean())
    else:
        x,observed,affine = download_chip(images[c['year']],c['x'],c['y'],c['epsg'])
        y = rasterize_labels(nearby(label_frames[c['year']],geo.bounds),affine,c['epsg'])
        known = rasterize_labels(nearby(history_frame,geo.bounds),affine,c['epsg'])
        # Não rotulado não significa floresta intacta. Avaliar negativos conservadores.
        valid = observed & ((y == 1) | ((known == 0) & (ndvi(x) > .75)))
        positive, frac = bool((y[valid] == 1).any()), float(valid.mean())
        np.savez_compressed(path,x=x,y=y,valid=valid,observed=observed,positive=positive)
    if frac < .25:
        return None
    return dict(c,path=str(path),positive=positive,valid_fraction=frac,chip_id=key)

positive_pool = [(year,f) for year,fs in labels.items() for f in fs]
if not positive_pool:
    raise RuntimeError('Nenhum rótulo positivo: verificar WFS, sem criar dados simulados.')
records, used = [], set()
failures = []
for round_id in range(12):
    missing = N_CHIPS-len(records)
    if missing <= 0:
        break
    candidates = []
    for i in range(max(32,int(missing*1.4))):
        if i % 4 != 0 or sum(r['positive'] for r in records) < N_CHIPS//2:
            year,f = positive_pool[int(rng.integers(len(positive_pool)))]
            p = shape(f['geometry']).representative_point()
            lon,lat = p.x,p.y
        else:
            year = int(rng.choice(ANOS))
            lon,lat = rng.uniform(BBOX[0],BBOX[2]),rng.uniform(BBOX[1],BBOX[3])
        c = candidate(lon,lat,year)
        key = (c['year'],c['x'],c['y'],c['epsg'])
        if key not in used:
            candidates.append(c)
            used.add(key)
    # executor.map mantém ordem determinística, independentemente da latência.
    def safe(c):
        try:
            return materialize(c)
        except Exception as exc:
            failures.append(str(exc)[:300])
            return None
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for result in pool.map(safe,candidates):
            if result is not None and len(records)<N_CHIPS:
                if not result['positive'] and sum(not r['positive'] for r in records)>=N_CHIPS//2:
                    continue
                records.append(result)
    print(f'Rodada {round_id+1}: {len(records)}/{N_CHIPS} chips; erros={len(failures)}')
if len(records)<N_CHIPS or sum(r['positive'] for r in records)<len(records)/2:
    raise RuntimeError(f'Meta de chips não atingida; cache preservado. Últimos erros: {failures[-3:]}')
assignment = split_blocks([r['bloco'] for r in records])
for r in records:
    r['split'] = assignment[r['bloco']]
for subset in ['train','val','test']:
    selected = [r for r in records if r['split']==subset]
    if not selected or not any(r['positive'] for r in selected):
        raise RuntimeError(f'Split {subset} sem positivos: não reportar métricas degeneradas.')
    print(subset,len(selected),'chips',len({r['bloco'] for r in selected}),'blocos')
(OUT/'split_manifest.json').write_text(json.dumps(records,indent=2),encoding='utf-8')
CONFIG.update(n_chips=len(records),positive_chips=sum(r['positive'] for r in records),download_errors=failures)
if SALVAR_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    shutil.copytree(CACHE,'/content/drive/MyDrive/lastro_chips',dirs_exist_ok=True)

## Treino: Prithvi via TerraTorch, fallback U-Net siamesa

In [ ]:
import gc
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Ative GPU em Ambiente de execução > Alterar tipo de ambiente e execute novamente.')
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:',torch.cuda.get_device_name(0),round(GPU_GB,1),'GB')

# Estatísticas publicadas pelo Prithvi-EO-2.0 para reflectância HLS.
MEAN = np.array([.108,.137,.151,.226,.240,.186],np.float32)
STD = np.array([.057,.065,.079,.103,.114,.104],np.float32)
CONFIG.update(normalization_mean=MEAN.tolist(),normalization_std=STD.tolist(),
              nir_proxy='B8 (10 m), em lugar de NIR narrow HLS; domain shift',gpu=torch.cuda.get_device_name(0))

class Chips(Dataset):
    def __init__(self, subset):
        self.rows = [r for r in records if r['split']==subset]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        with np.load(self.rows[i]['path']) as z:
            x,y,valid = z['x'].copy(),z['y'].copy(),z['valid'].copy()
            observed = z['observed'].copy()
        x = (x-np.tile(MEAN,2)[:,None,None])/np.tile(STD,2)[:,None,None]
        x[:,~observed] = 0
        return torch.from_numpy(x),torch.from_numpy(y).float(),torch.from_numpy(valid)

datasets = {s:Chips(s) for s in ['train','val','test']}

class PrithviSegmenter(nn.Module):
    """Backbone TerraTorch multitemporal + decoder convolucional de dois frames."""
    def __init__(self, variant):
        super().__init__()
        from terratorch.registry import BACKBONE_REGISTRY
        self.backbone = BACKBONE_REGISTRY.build(
            variant, pretrained=True, num_frames=2,
            bands=['BLUE','GREEN','RED','NIR_NARROW','SWIR_1','SWIR_2'])
        dim = self.backbone.out_channels[-1]
        self.decoder = nn.Sequential(nn.Conv2d(dim*2,128,1),nn.GroupNorm(8,128),nn.GELU(),
                                     nn.Conv2d(128,64,3,padding=1),nn.GELU(),nn.Conv2d(64,1,1))
        # Fine-tune dos últimos 2/4 blocos e decoder reduz memória e tempo na T4.
        for p in self.backbone.parameters():
            p.requires_grad = False
        blocks = getattr(self.backbone,'blocks',None)
        if blocks is None:
            blocks = getattr(getattr(self.backbone,'encoder',None),'blocks',None)
        if blocks is None:
            raise RuntimeError('Backbone sem blocks acessíveis para fine-tuning parcial.')
        for block in blocks[-(2 if MODO_RAPIDO else 4):]:
            for p in block.parameters():
                p.requires_grad = True
    def forward(self,x):
        # CHW concatenado -> B,C,T,H,W (não B,T,C,H,W).
        t = x.reshape(x.shape[0],2,6,*x.shape[-2:]).transpose(1,2)
        features = self.backbone(t)[-1]
        h = x.shape[-2]//16
        if features.ndim != 3:
            raise RuntimeError(f'Formato inesperado do backbone: {features.shape}')
        if features.shape[1] == 2*h*h+1:
            features = features[:,1:]
        if features.shape[1] != 2*h*h:
            raise RuntimeError(f'Tokens temporais inesperados: {features.shape}')
        features = features.reshape(x.shape[0],2,h,h,-1).permute(0,1,4,2,3).flatten(1,2)
        return F.interpolate(self.decoder(features),size=x.shape[-2:],mode='bilinear',align_corners=False)[:,0]

class SiameseUNet(nn.Module):
    """Dois ramos de seis bandas, ResNet34 compartilhada e fusão pré/pós."""
    def __init__(self):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net = smp.Unet(encoder_name='resnet34',encoder_weights='imagenet',in_channels=6,classes=1)
        self.fuse = nn.ModuleList([nn.Conv2d(2*c,c,1) for c in self.net.encoder.out_channels])
    def forward(self,x):
        a,b = self.net.encoder(x[:,:6]),self.net.encoder(x[:,6:])
        fused = [layer(torch.cat([p,q],1)) for layer,p,q in zip(self.fuse,a,b)]
        return self.net.segmentation_head(self.net.decoder(fused))[:,0]

def loss_fn(logits,y,valid):
    if not valid.any():
        return logits.sum()*0
    bce = F.binary_cross_entropy_with_logits(logits[valid],y[valid])
    p,t = logits.sigmoid()[valid],y[valid]
    dice = 1-(2*(p*t).sum()+1)/(p.sum()+t.sum()+1)
    return bce+dice

def fit(model,batch):
    model = model.to(DEVICE)
    loaders = {s:DataLoader(ds,batch_size=batch,shuffle=(s=='train'),num_workers=0,
                           generator=torch.Generator().manual_seed(SEED)) for s,ds in datasets.items()}
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, log = float('inf'),0,[]
    for epoch in range(EPOCAS):
        losses = {}
        for subset in ['train','val']:
            model.train(subset=='train')
            total = count = 0
            for x,y,valid in loaders[subset]:
                x,y,valid = x.to(DEVICE),y.to(DEVICE),valid.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(subset=='train'),torch.autocast('cuda',dtype=torch.float16):
                    logits = model(x)
                    loss = loss_fn(logits,y,valid)
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss não finita; interrompido para evitar métricas inválidas.')
                if subset=='train':
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1)
                    scaler.step(optimizer)
                    scaler.update()
                total += loss.item()*len(x)
                count += len(x)
            losses[subset] = total/count
        log.append(dict(epoch=epoch+1,**losses))
        print(f'Época {epoch+1}/{EPOCAS}: {losses}',flush=True)
        if losses['val'] < best-1e-4:
            best,stale = losses['val'],0
            torch.save(model.state_dict(),ROOT/'lastro_best.pt')
        else:
            stale += 1
        if stale >= 3:
            print('Early stopping (paciência 3).')
            break
    model.load_state_dict(torch.load(ROOT/'lastro_best.pt',map_location=DEVICE,weights_only=True))
    return model.eval(),log

attempts = []
variants = ['prithvi_eo_v2_100_tl'] if GPU_GB<20 else ['prithvi_eo_v2_300_tl','prithvi_eo_v2_100_tl']
variants += ['unet-siamesa']
model = None
for variant in variants:
    try:
        print('Tentando',variant,flush=True)
        torch.manual_seed(SEED)
        model = SiameseUNet() if variant=='unet-siamesa' else PrithviSegmenter(variant)
        model,train_log = fit(model,batch=4 if variant=='unet-siamesa' else 2)
        MODEL_NAME = 'unet-siamesa' if variant=='unet-siamesa' else 'prithvi-eo-2.0'
        break
    except Exception as exc:
        attempts.append(dict(model=variant,error=f'{type(exc).__name__}: {exc}'))
        print('Fallback:',attempts[-1],flush=True)
        del model
        model = None
        gc.collect()
        torch.cuda.empty_cache()
if model is None:
    raise RuntimeError(f'Nenhum modelo treinou: {attempts}')
CONFIG.update(modelo=MODEL_NAME,variant=variant,fallbacks=attempts,
              epocas_executadas=len(train_log),treino=train_log,lr=1e-4,
              fine_tune='últimos 2/4 blocos Prithvi ou encoder completo ResNet34',early_stopping_patience=3)

## Avaliação e comparadores nos mesmos pixels

In [ ]:
def metrics(cm):
    tn,fp,fn,tp = map(int,cm.ravel())
    ratio = lambda a,b: a/b if b else 0.0
    return dict(iou=ratio(tp,tp+fp+fn),f1=ratio(2*tp,2*tp+fp+fn),
                precisao=ratio(tp,tp+fp),recall=ratio(tp,tp+fn),
                matriz_confusao=cm.tolist(),n_pixels_teste=int(cm.sum()))

@torch.inference_mode()
def predict_array(x,valid):
    normalized = (x-np.tile(MEAN,2)[:,None,None])/np.tile(STD,2)[:,None,None]
    normalized[:,~valid] = 0
    with torch.autocast('cuda',dtype=torch.float16):
        prob = model(torch.from_numpy(normalized[None]).to(DEVICE)).sigmoid()[0].float().cpu().numpy()
    return prob

def delta_ndvi(x):
    return ndvi(x)-ndvi(x,6)  # positivo = perda de vigor

def lgb_features(x):
    indices = []
    for offset in (0,6):
        nir = x[offset+3]
        indices.append(np.stack([ndvi(x,offset),
                                 (nir-x[offset+5])/(nir+x[offset+5]+1e-6),
                                 (nir-x[offset+4])/(nir+x[offset+4]+1e-6)]))
    return np.concatenate([x,*indices,indices[0]-indices[1]],axis=0).transpose(1,2,0)

def sample_pixels(subset,max_per_chip=1000):
    xs,ys,ds = [],[],[]
    rng = np.random.default_rng(SEED)
    for r in records:
        if r['split'] != subset:
            continue
        with np.load(r['path']) as z:
            ids = np.flatnonzero(z['valid'])
            ids = rng.choice(ids,size=min(len(ids),max_per_chip),replace=False)
            xs.append(lgb_features(z['x']).reshape(-1,21)[ids])
            ys.append(z['y'].ravel()[ids])
            ds.append(delta_ndvi(z['x']).ravel()[ids])
    return np.concatenate(xs),np.concatenate(ys),np.concatenate(ds)

import lightgbm as lgb
xt,yt,dt = sample_pixels('train')
xv,yv,_ = sample_pixels('val')
otsu = max(0.,float(threshold_otsu(dt)))  # limiar aprendido apenas no treino
lgbm = lgb.LGBMClassifier(n_estimators=350,num_leaves=31,learning_rate=.05,
                          random_state=SEED,n_jobs=2,verbosity=-1,deterministic=True,force_col_wise=True)
lgbm.fit(xt,yt,eval_set=[(xv,yv)],callbacks=[lgb.early_stopping(25,verbose=False)])
pv = lgbm.predict_proba(xv)[:,1]
thresholds = np.arange(.1,.91,.05)
lgb_threshold = float(max(thresholds,key=lambda t:metrics(confusion_matrix(yv,pv>=t,labels=[0,1]))['f1']))
lgbm.booster_.save_model(str(OUT/'lgbm_mesmos_chips.txt'))
cms = {key:np.zeros((2,2),dtype=np.int64) for key in ['dl','baseline','lightgbm']}
examples = 0
for r in records:
    if r['split']!='test':
        continue
    with np.load(r['path']) as z:
        x,y,valid = z['x'],z['y'],z['valid']
        prob = predict_array(x,z['observed'])
        base = delta_ndvi(x)>otsu
        # Inferir só pixels observáveis, em lotes pequenos.
        features = lgb_features(x)[valid]
        plgb = lgbm.predict_proba(features)[:,1]>=lgb_threshold
        for key,p in [('dl',prob[valid]>=.5),('baseline',base[valid]),('lightgbm',plgb)]:
            cms[key] += confusion_matrix(y[valid],p,labels=[0,1])
        if examples<4:
            fig,axs = plt.subplots(1,4,figsize=(14,4))
            panels = [np.clip(x[[2,1,0]].transpose(1,2,0)/.3,0,1),
                      np.clip(x[[8,7,6]].transpose(1,2,0)/.3,0,1),
                      np.where(valid,y,np.nan),np.where(valid,prob,np.nan)]
            for ax,p,title in zip(axs,panels,['Pré','Pós','Rótulo (vazio = ignorado)','Probabilidade']):
                ax.imshow(p,vmin=0,vmax=1)
                ax.set_title(title)
                ax.axis('off')
            fig.suptitle(f"Teste espacial: {r['bloco']} / {r['year']}")
            fig.tight_layout()
            fig.savefig(OUT/f'exemplo_{examples+1}.png',dpi=140)
            plt.close(fig)
            examples += 1
PROTOCOL = ('Blocos UTM SIRGAS2000 de 25 km, seed 42, 70/15/15 por bloco; '
            'todos os anos de um bloco no mesmo split; chips inteiros dentro do bloco; '
            'AOIs de demonstração excluídas. Métricas de pixel apenas no teste espacial. '
            'Pixels sem observação e negativos sem NDVI pré > 0,75 ou com histórico de alertas ignorados. '
            'DL limiar fixo 0,5; Otsu ajustado no treino; LightGBM limiar ajustado na validação. '
            'Matriz [[TN,FP],[FN,TP]]. Amostragem enriquecida em positivos, não estima prevalência regional.')
OBS = ['Resultados experimentais: indícios para priorizar fiscalização; não constituem prova.',
       'PRODES/DETER não captam todas as pequenas clareiras; ausência de rótulo não garante floresta intacta.',
       'PRODES usa ano de monitoramento e DETER out/set; mediana jun/set pode diluir mudanças e não coincide exatamente com eventos.',
       'Proxy B8 para NIR narrow HLS e resolução 10 m implicam mudança de domínio do Prithvi.',
       'LightGBM abaixo é refeito nestes chips; métricas do detector local com 5 folds/5 km não são diretamente comparáveis.',
       'Tempos de 30 min/2 h são metas, dependem da GPU, rede, WFS, GEE e quantidade de nuvens.']
report = dict(nome=MODEL_NAME,**metrics(cms['dl']),obs=OBS,protocolo=PROTOCOL,
              baseline_mesmos_chips=dict(nome='delta-NDVI + Otsu',threshold=otsu,**metrics(cms['baseline'])),
              lightgbm_mesmos_chips=dict(nome='lightgbm',threshold=lgb_threshold,**metrics(cms['lightgbm'])))
(OUT/'metrics_dl.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
CONFIG.update(otsu_threshold=otsu,lgbm_threshold=lgb_threshold,lgbm_features=21,
              lgbm_params=lgbm.get_params(),protocolo=PROTOCOL)
print(json.dumps(report,indent=2,ensure_ascii=False))

## AOIs: GeoTIFF em grade UTM, janelas sobrepostas e pesos nas bordas

In [ ]:
from rasterio.windows import Window

def window_starts(length,step=168):
    if length<=SIZE:
        return [0]
    return sorted(set(list(range(0,length-SIZE+1,step))+[length-SIZE]))

def infer_aoi(aoi):
    epsg = utm_epsg((aoi['bbox'][0]+aoi['bbox'][2])/2)
    geometry = geom_transform(Transformer.from_crs(4326,epsg,always_xy=True).transform,box(*aoi['bbox']))
    xmin,ymin,xmax,ymax = geometry.bounds
    left,top = math.floor(xmin/RES)*RES,math.ceil(ymax/RES)*RES
    width,height = math.ceil((xmax-left)/RES),math.ceil((top-ymin)/RES)
    if width*height>100_000_000:
        raise ValueError('AOI maior que 100 milhões de pixels: dividir a AOI em partes.')
    affine = from_origin(left,top,RES,RES)
    image = stack_image(aoi['pre'],aoi['pos'],ee.Geometry.Rectangle(aoi['bbox']))
    acc = np.zeros((height,width),np.float32)
    weights = np.zeros_like(acc)
    weight = np.maximum(np.outer(np.hanning(SIZE),np.hanning(SIZE)),.01).astype('float32')
    coordinates = [(r,c) for r in window_starts(height) for c in window_starts(width)]
    def fetch(rc):
        r,c = rc
        return rc,download_chip(image,left+c*RES,top-r*RES,epsg)
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        # map pode baixar adiantado; limite de memória pela quantidade da AOI.
        for i,((r,c),(x,valid,_)) in enumerate(pool.map(fetch,coordinates)):
            p = predict_array(x,valid)
            h,w = min(SIZE,height-r),min(SIZE,width-c)
            weighted = weight[:h,:w]*valid[:h,:w]
            acc[r:r+h,c:c+w] += p[:h,:w]*weighted
            weights[r:r+h,c:c+w] += weighted
            if (i+1)%10==0 or i==len(coordinates)-1:
                print(aoi['autex'],i+1,'/',len(coordinates),'janelas')
    observed = weights>0
    prob = np.full((height,width),-9999,np.float32)
    prob[observed] = acc[observed]/weights[observed]
    mask = np.full((height,width),255,np.uint8)
    mask[observed] = (prob[observed]>=.5).astype(np.uint8)
    name = re.sub(r'[^A-Za-z0-9_-]','_',aoi['autex'])
    for prefix,arr,nodata in [('prob',prob,-9999),('mask',mask,255)]:
        with rasterio.open(OUT/f'{prefix}_{name}.tif','w',driver='GTiff',
                           height=height,width=width,count=1,dtype=arr.dtype,
                           crs=f'EPSG:{epsg}',transform=affine,nodata=nodata,
                           compress='deflate',tiled=True) as dst:
            dst.write(arr,1)
            dst.update_tags(modelo=MODEL_NAME,autex=aoi['autex'],aviso='Triagem experimental; não constitui prova')
    return dict(autex=aoi['autex'],epsg=epsg,width=width,height=height,
                valid_fraction=float(observed.mean()),tiles=len(coordinates))

CONFIG['aoi_results'] = [infer_aoi(a) for a in AOIS]

## Empacotar e baixar

In [ ]:
import shutil, importlib.metadata
CONFIG['versions'] = {p:importlib.metadata.version(p) for p in
                      ['torch','numpy','earthengine-api','rasterio','lightgbm','segmentation-models-pytorch']}
try:
    CONFIG['versions']['terratorch'] = importlib.metadata.version('terratorch')
except importlib.metadata.PackageNotFoundError:
    pass
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
(OUT/'LEIA-ME.txt').write_text('\n'.join(OBS)+'\n'+PROTOCOL,encoding='utf-8')
archive = shutil.make_archive(str(ROOT/'lastro_dl_resultados'),'zip',OUT)
if SALVAR_DRIVE:
    shutil.copy2(archive,'/content/drive/MyDrive/lastro_dl_resultados.zip')
print('Resultado:',archive,'; modelo:',MODEL_NAME)
from google.colab import files
files.download(archive)

### Referências e limites
- [Prithvi EO / TerraTorch](https://torchgeo.org/terratorch/1.2.10/guide/prithvi_eo/)
- [Prithvi 300M-TL](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL)
- [TerraBrasilis WFS](https://terrabrasilis.dpi.inpe.br/geoserver/ows)
- [GEE computePixels](https://developers.google.com/earth-engine/apidocs/ee-data-computepixels)

Rótulos de monitoramento são incompletos, sobretudo para manejo seletivo. O conjunto foi enriquecido em positivos e não mede a prevalência regional. O LightGBM incluído é treinado nestes mesmos chips; comparar diretamente com o detector local exige que este reutilize `split_manifest.json`. As métricas locais de cinco folds/5 km constituem outro experimento. O zip inclui protocolo, versões, erros de fallback e configuração. A máscara de observação usada como entrada é independente dos rótulos; a máscara de avaliação inclui apenas positivos conhecidos e negativos conservadores.